# multi_1m_jma_atr_042

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (44).ipynb`

| Field | Value |
|---|---|
| Section | `strategies` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `1m` |
| Code cells | 8 |
| Detected functions | resample_1hr_custom, __init__, compute_atr, compute_jma_and_signals, compute_atr_max, map_signals_to_1min, backtest_1min_exits_numba, run_backtest, calculate_metrics, normalize_metric, calculate_composite_score, __init__ |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** JMA + ATR crossover (detected)
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data_1m = pd.read_parquet('/content/drive/MyDrive/Copy of Binance_Crypto_T5_2020_25.parquet')
data_1m.rename(columns={
    'Datetime': 'Date',
    'Open': 'open',
    'Close': 'close',
    'Low': 'low',
    'High': 'high',
    'Volume': 'volume'
}, inplace=True)

data_1m

In [ ]:
data_1m['Ticker'].unique()

In [ ]:
# Convert 'Date' column to datetime if it isn't already
data_1m['Date'] = pd.to_datetime(data_1m['Date'])

# Define the date range
start_date = '2021-01-01'
end_date = '2025-12-31'

# Define the tickers you want to filter
selected_tickers = ['SOLUSDT']  # Add tickers you want

# Apply filtering
data_1m = data_1m[
    (data_1m["Date"] >= start_date) &
    (data_1m["Date"] <= end_date) &
    (data_1m["Ticker"].isin(selected_tickers))
]

# Display filtered result
data = data_1m.copy()
data

In [ ]:
import pandas as pd

# Ensure Date is datetime
data['Date'] = pd.to_datetime(data['Date'])

# Custom resampling function with offset
def resample_1hr_custom(group):
    ticker = group['Ticker'].iloc[0]
    group = group.set_index('Date')
    resampled = group.resample('4h').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
    }).dropna().reset_index()
    resampled['Ticker'] = ticker
    return resampled

# Apply per ticker
data_1hr = data.groupby('Ticker').apply(resample_1hr_custom).reset_index(drop=True)

# Reorder columns
data_1hr = data_1hr[['Ticker', 'Date', 'open', 'high', 'low', 'close']]

data = data_1hr
data

### **INDICATOR CODE**

# **BACKTESTING CODE**

In [ ]:
import pandas as pd
import numpy as np
from numba import njit

# ===============================
# ATR Calculator (Vectorized)
# ===============================
class ATRIndicator:
    """Average True Range (ATR) Calculator"""
    def __init__(self, high_col='high', low_col='low', close_col='close'):
        self.high_col = high_col
        self.low_col = low_col
        self.close_col = close_col

    def compute_atr(self, df, period=14, shift=1):
        high = df[self.high_col].values
        low = df[self.low_col].values
        close = df[self.close_col].values

        # Vectorized TR calculation
        prev_close = np.roll(close, 1)
        prev_close[0] = close[0]

        tr = np.maximum(high - low,
             np.maximum(np.abs(high - prev_close),
                       np.abs(low - prev_close)))

        # Fast rolling mean
        atr = pd.Series(tr).rolling(window=period, min_periods=1).mean().shift(shift).values
        return atr

# ===============================
# JMA + Signals (Optimized)
# ===============================
def compute_jma_and_signals(df, smooth_length=14, normalization_period=108,
                            upper_threshold=2.25, lower_threshold=-1.25,
                            rs_period=3, rs_percentile=10):
    df = df.copy()
    price = df['close'].values

    # JMA calculation (vectorized)
    alpha = 2.0 / (smooth_length + 1)
    jma_values = pd.Series(price).ewm(alpha=alpha, adjust=False).mean().values
    df['JMA'] = jma_values

    # Normalization (vectorized)
    jma_shifted = np.roll(jma_values, 1)
    jma_shifted[0] = jma_values[0]

    rolling_mean = pd.Series(jma_shifted).rolling(normalization_period).mean().values
    rolling_std = pd.Series(jma_shifted).rolling(normalization_period).std().values
    df['Normalized_JMA'] = (jma_shifted - rolling_mean) / rolling_std

    # Rogers-Satchell Volatility (vectorized)
    o = df['open'].replace(0, np.nan).values
    h = df['high'].values
    l = df['low'].values
    c = df['close'].replace(0, np.nan).values

    with np.errstate(divide='ignore', invalid='ignore'):
        rs_base = (np.log(h / c) * np.log(h / o) + np.log(l / c) * np.log(l / o))

    rs_vol = pd.Series(rs_base).rolling(window=rs_period).mean().shift(1).values * 1e6
    df['RS_Volatility'] = rs_vol

    rs_thresh = pd.Series(rs_vol).shift(1).rolling(normalization_period).quantile(rs_percentile / 100.0).values
    df['RS_Threshold'] = rs_thresh

    # Signals (vectorized)
    norm_jma = df['Normalized_JMA'].values
    rs_vol = df['RS_Volatility'].values
    rs_thresh = df['RS_Threshold'].values

    signals = np.zeros(len(df), dtype=np.int8)
    signals[(norm_jma > upper_threshold) & (rs_vol > rs_thresh)] = 1
    signals[(norm_jma < lower_threshold) & (rs_vol > rs_thresh)] = 2
    df['Signal'] = signals

    return df

# ===============================
# ATR_max computation (Vectorized)
# ===============================
def compute_atr_max(group, atr_mult=0.1):
    atr_14 = group['ATR_14'].values
    atr_5 = group['ATR_5'].values

    atr_14_shifted = np.roll(atr_14, 1)
    atr_5_shifted = np.roll(atr_5, 1)
    atr_14_shifted[0] = atr_14[0]
    atr_5_shifted[0] = atr_5[0]

    atr_14_max = pd.Series(atr_14_shifted).rolling(20, min_periods=1).max().values * atr_mult
    atr_5_max = pd.Series(atr_5_shifted).rolling(20, min_periods=1).max().values * atr_mult

    return np.maximum(atr_14_max, atr_5_max)

# ===============================
# Signal Mapping (Optimized)
# ===============================
def map_signals_to_1min(data_1min, data_4h):
    data_1min = data_1min.copy()
    data_1min['Date'] = pd.to_datetime(data_1min['Date'])
    data_1min['Date_4H'] = data_1min['Date'].dt.floor('4H')

    signal_map = data_4h[['Date', 'Ticker', 'Signal', 'ATR_max']].rename(columns={'Date': 'Date_4H'})

    # Use merge with sorted data for faster join
    data_1min = data_1min.merge(signal_map, on=['Date_4H', 'Ticker'], how='left')

    # Vectorized ffill per group
    data_1min['Signal'] = data_1min.groupby('Ticker', group_keys=False)['Signal'].ffill().fillna(0).astype(np.int8)
    data_1min['ATR_max'] = data_1min.groupby('Ticker', group_keys=False)['ATR_max'].ffill()

    return data_1min

# ===============================
# Backtest Function (Numba JIT)
# ===============================
@njit
def backtest_1min_exits_numba(dates, opens, highs, lows, closes, signals, atr_maxs, dates_4h,
                                capital, fee_rate, slippage_rate, entry_sl_pct):
    """
    Numba-compiled backtest loop for maximum speed
    Returns arrays that will be converted to DataFrame
    """
    n = len(dates)

    # Pre-allocate result arrays (max possible trades)
    max_trades = n // 10  # estimate
    entry_times = np.empty(max_trades, dtype=np.int64)
    exit_times = np.empty(max_trades, dtype=np.int64)
    entry_prices = np.empty(max_trades, dtype=np.float64)
    exit_prices = np.empty(max_trades, dtype=np.float64)
    pnls = np.empty(max_trades, dtype=np.float64)
    cumulative_pnls = np.empty(max_trades, dtype=np.float64)
    directions = np.empty(max_trades, dtype=np.int8)
    entry_4h_times = np.empty(max_trades, dtype=np.int64)
    exit_4h_times = np.empty(max_trades, dtype=np.int64)
    bars_since_entries = np.empty(max_trades, dtype=np.int32)
    exit_reasons = np.empty(max_trades, dtype=np.int8)  # 0=entry_sl, 1=trailing_sl

    trade_count = 0
    cumulative_pnl = 0.0
    position_open = False
    entry_price = sl_price = quantity = 0.0
    entry_time = entry_4h_time = last_exit_4h_time = 0
    direction = bars_since_entry = 0

    prev_4h_time = 0
    completed_4h_count = 0
    max_4h_bars = 10000
    completed_4h_times = np.empty(max_4h_bars, dtype=np.int64)
    completed_4h_closes = np.empty(max_4h_bars, dtype=np.float64)
    completed_4h_atrs = np.empty(max_4h_bars, dtype=np.float64)

    for i in range(n):
        time = dates[i]
        open_ = opens[i]
        high = highs[i]
        low = lows[i]
        close = closes[i]
        signal = signals[i]
        atr_max = atr_maxs[i]
        time_4h = dates_4h[i]

        is_new_4h_bar = (prev_4h_time > 0 and time_4h > prev_4h_time)

        # Store completed 4H bar
        if is_new_4h_bar and prev_4h_time > 0 and i > 0 and completed_4h_count < max_4h_bars:
            completed_4h_times[completed_4h_count] = prev_4h_time
            completed_4h_closes[completed_4h_count] = closes[i-1]
            completed_4h_atrs[completed_4h_count] = atr_maxs[i-1]
            completed_4h_count += 1

        # ENTRY
        if not position_open and signal in (1, 2) and is_new_4h_bar and time_4h != last_exit_4h_time:
            direction = 1 if signal == 1 else -1
            entry_price = open_ * (1 + slippage_rate * direction)
            quantity = capital / entry_price
            entry_time = time
            entry_4h_time = time_4h
            position_open = True
            bars_since_entry = 0
            sl_price = entry_price * (1 - entry_sl_pct * direction)

        # STOP LOSS UPDATE
        if position_open and is_new_4h_bar:
            bars_since_entry += 1

            if bars_since_entry == 1:
                # Find entry bar in completed bars
                for j in range(completed_4h_count):
                    if completed_4h_times[j] == entry_4h_time:
                        sl_price = completed_4h_closes[j] - completed_4h_atrs[j] * direction
                        break

            elif bars_since_entry >= 2:
                # Find previous bar
                for j in range(completed_4h_count):
                    if completed_4h_times[j] == prev_4h_time:
                        new_sl = completed_4h_closes[j] - completed_4h_atrs[j] * direction
                        if direction == 1:
                            sl_price = max(sl_price, new_sl)
                        else:
                            sl_price = min(sl_price, new_sl)
                        break

        # CHECK STOP LOSS
        if position_open and entry_time != time:
            sl_hit = (direction == 1 and low <= sl_price) or (direction == -1 and high >= sl_price)

            if sl_hit:
                pnl = (sl_price - entry_price) * quantity * direction
                fee = fee_rate * (entry_price + sl_price) * quantity
                net_pnl = pnl - fee
                cumulative_pnl += net_pnl

                if trade_count < max_trades:
                    entry_times[trade_count] = entry_time
                    exit_times[trade_count] = time
                    entry_prices[trade_count] = entry_price
                    exit_prices[trade_count] = sl_price
                    pnls[trade_count] = net_pnl
                    cumulative_pnls[trade_count] = cumulative_pnl
                    directions[trade_count] = direction
                    entry_4h_times[trade_count] = entry_4h_time
                    exit_4h_times[trade_count] = time_4h
                    bars_since_entries[trade_count] = bars_since_entry
                    exit_reasons[trade_count] = 0 if bars_since_entry == 0 else 1
                    trade_count += 1

                position_open = False
                last_exit_4h_time = time_4h
                bars_since_entry = 0

        prev_4h_time = time_4h

    # Return only filled portion of arrays
    return (entry_times[:trade_count], exit_times[:trade_count],
            entry_prices[:trade_count], exit_prices[:trade_count],
            pnls[:trade_count], cumulative_pnls[:trade_count],
            directions[:trade_count], entry_4h_times[:trade_count],
            exit_4h_times[:trade_count], bars_since_entries[:trade_count],
            exit_reasons[:trade_count])

# ===============================
# Main Pipeline (Optimized)
# ===============================
def run_backtest(data_4h, data_1m, capital=100000, fee_rate=0.0001,
                 slippage_rate=0.0001, entry_sl_pct=0.02):
    """Run complete backtest pipeline"""

    print("Processing 4H data...")
    data_4h = data_4h.copy()
    data_4h['Date'] = pd.to_datetime(data_4h['Date'])

    # Sort for faster processing
    data_4h = data_4h.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    # Compute indicators per ticker (parallelizable if needed)
    groups = []
    for ticker, group in data_4h.groupby('Ticker'):
        group = compute_jma_and_signals(group)
        groups.append(group)
    data_4h = pd.concat(groups, ignore_index=True)

    # ATR calculation
    atr_calc = ATRIndicator()
    for period in [14, 5]:
        col_name = f'ATR_{period}'
        data_4h[col_name] = 0.0
        for ticker, group in data_4h.groupby('Ticker'):
            idx = group.index
            data_4h.loc[idx, col_name] = atr_calc.compute_atr(group, period=period, shift=1)

    # ATR_max
    groups = []
    for ticker, group in data_4h.groupby('Ticker'):
        group['ATR_max'] = compute_atr_max(group)
        groups.append(group)
    data_4h = pd.concat(groups, ignore_index=True)

    # Shift signals
    data_4h['Signal'] = data_4h.groupby('Ticker', group_keys=False)['Signal'].shift(1)
    data_4h = data_4h.dropna(subset=['Signal'])
    data_4h['Signal'] = data_4h['Signal'].astype(np.int8)

    print("Mapping signals to 1min data...")
    data_1min = map_signals_to_1min(data_1m, data_4h)
    data_1min = data_1min.sort_values(['Ticker', 'Date']).reset_index(drop=True)

    print("Running backtest...")
    all_trades = []

    for ticker, group in data_1min.groupby('Ticker'):
        # Convert to numpy arrays for numba
        dates = group['Date'].values.astype('datetime64[ns]').astype(np.int64)
        opens = group['open'].values.astype(np.float64)
        highs = group['high'].values.astype(np.float64)
        lows = group['low'].values.astype(np.float64)
        closes = group['close'].values.astype(np.float64)
        signals = group['Signal'].values.astype(np.int8)
        atr_maxs = group['ATR_max'].values.astype(np.float64)
        dates_4h = group['Date_4H'].values.astype('datetime64[ns]').astype(np.int64)

        result = backtest_1min_exits_numba(
            dates, opens, highs, lows, closes, signals, atr_maxs, dates_4h,
            capital, fee_rate, slippage_rate, entry_sl_pct
        )

        # Unpack results
        (entry_times, exit_times, entry_prices, exit_prices,
         pnls, cumulative_pnls, directions, entry_4h_times,
         exit_4h_times, bars_since_entries, exit_reasons) = result

        # Convert back to readable format
        for i in range(len(entry_times)):
            all_trades.append((
                ticker,
                pd.Timestamp(entry_times[i]),
                pd.Timestamp(exit_times[i]),
                entry_prices[i],
                exit_prices[i],
                pnls[i],
                "Entry SL (2%)" if exit_reasons[i] == 0 else "Trailing SL",
                cumulative_pnls[i],
                "long" if directions[i] == 1 else "short",
                pd.Timestamp(entry_4h_times[i]),
                pd.Timestamp(exit_4h_times[i]),
                bars_since_entries[i]
            ))

    trades_df = pd.DataFrame(all_trades, columns=[
        'Ticker', 'Entry Time', 'Exit Time', 'Entry Price', 'Exit Price',
        'PnL', 'Exit Reason', 'Cumulative PnL', 'Direction',
        'Entry 4H Time', 'Exit 4H Time', 'Bars Since Entry'
    ])

    print(f"Complete! Total trades: {len(trades_df)}")
    return trades_df

# Run backtest
trade_df = run_backtest(data, data_1m)
trade_df

In [ ]:
"""
WALK-FORWARD OPTIMIZATION SYSTEM - Fixed for Numba Implementation
===================================================================

Compatible with the updated Numba-compiled backtest function.
Requires: ATRIndicator, compute_jma_and_signals, compute_atr_max,
         map_signals_to_1min, backtest_1min_exits_numba
"""

import pandas as pd
import numpy as np
from itertools import product
from tqdm import tqdm
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

# ===============================
# PERFORMANCE METRICS CALCULATOR
# ===============================

def calculate_metrics(tradebook, initial_capital=100000):
    """Calculate comprehensive performance metrics from a tradebook."""
    if len(tradebook) == 0:
        return {
            'total_trades': 0,
            'profit_to_drawdown': 0,
            'equity_r2': 0,
            'monthly_consistency': 0,
            'daily_drawdown': 0,
            'total_pnl': 0,
            'win_rate': 0,
            'avg_pnl_per_trade': 0,
            'max_drawdown': 0
        }

    total_pnl = tradebook['PnL'].sum()
    total_trades = len(tradebook)
    wins = (tradebook['PnL'] > 0).sum()
    win_rate = wins / total_trades if total_trades > 0 else 0
    avg_pnl_per_trade = total_pnl / total_trades if total_trades > 0 else 0

    tradebook = tradebook.sort_values('Exit Time').reset_index(drop=True)
    tradebook['Cumulative PnL'] = tradebook['PnL'].cumsum()

    running_max = tradebook['Cumulative PnL'].expanding().max()
    drawdown = running_max - tradebook['Cumulative PnL']
    max_drawdown = drawdown.max()

    profit_to_drawdown = total_pnl / max_drawdown if max_drawdown > 0 else 0

    if len(tradebook) >= 2:
        x = np.arange(len(tradebook))
        y = tradebook['Cumulative PnL'].values
        try:
            equity_r2 = r2_score(y, np.polyval(np.polyfit(x, y, 1), x))
            equity_r2 = max(0, equity_r2)
        except:
            equity_r2 = 0
    else:
        equity_r2 = 0

    tradebook['Exit Month'] = pd.to_datetime(tradebook['Exit Time']).dt.to_period('M')
    monthly_pnl = tradebook.groupby('Exit Month')['PnL'].sum()

    if len(monthly_pnl) >= 2:
        positive_months = (monthly_pnl > 0).sum()
        monthly_consistency = positive_months / len(monthly_pnl)
    else:
        monthly_consistency = 0

    tradebook['Exit Date'] = pd.to_datetime(tradebook['Exit Time']).dt.date
    daily_pnl = tradebook.groupby('Exit Date')['PnL'].sum()
    daily_cumsum = daily_pnl.cumsum()
    daily_running_max = daily_cumsum.expanding().max()
    daily_drawdown_series = daily_running_max - daily_cumsum
    avg_daily_drawdown = daily_drawdown_series.mean()

    daily_drawdown_score = 1 - min(1, avg_daily_drawdown / initial_capital)

    return {
        'total_trades': total_trades,
        'profit_to_drawdown': profit_to_drawdown,
        'equity_r2': equity_r2,
        'monthly_consistency': monthly_consistency,
        'daily_drawdown': daily_drawdown_score,
        'total_pnl': total_pnl,
        'win_rate': win_rate,
        'avg_pnl_per_trade': avg_pnl_per_trade,
        'max_drawdown': max_drawdown
    }


def normalize_metric(value, metric_name, all_values=None):
    """Normalize metrics to 0-1 range for fair comparison."""
    if metric_name == 'profit_to_drawdown':
        if all_values is not None and len(all_values) > 0:
            max_val = np.percentile(all_values, 95)
            return min(1.0, value / max_val) if max_val > 0 else 0
        else:
            return min(1.0, value / 5.0)
    return value


def calculate_composite_score(metrics, weights, all_profit_to_drawdown=None):
    """Calculate weighted composite score combining multiple metrics."""
    score = (
        weights['profit_to_drawdown'] * normalize_metric(
            metrics['profit_to_drawdown'],
            'profit_to_drawdown',
            all_profit_to_drawdown
        ) +
        weights['equity_r2'] * metrics['equity_r2'] +
        weights['monthly_consistency'] * metrics['monthly_consistency'] +
        weights['daily_drawdown'] * metrics['daily_drawdown']
    )
    return score


# ===============================
# WALK-FORWARD OPTIMIZER CLASS
# ===============================

class WalkForwardOptimizer:
    """
    Walk-forward optimization system using Numba-compiled backtest.

    Requires both 4H and 1-minute data.
    """

    def __init__(self, data_4h, data_1m, initial_capital=100000,
                 fee_rate=0.0005, slippage_rate=0.0005):
        """
        Initialize optimizer with both timeframes.

        Args:
            data_4h: 4-hour OHLC data for signals
            data_1m: 1-minute OHLC data for exits
            initial_capital: Starting capital
            fee_rate: Trading fee rate
            slippage_rate: Slippage rate
        """
        self.raw_data_4h = data_4h.copy()
        self.raw_data_4h['Date'] = pd.to_datetime(self.raw_data_4h['Date'])

        self.raw_data_1m = data_1m.copy()
        self.raw_data_1m['Date'] = pd.to_datetime(self.raw_data_1m['Date'])

        self.initial_capital = initial_capital
        self.fee_rate = fee_rate
        self.slippage_rate = slippage_rate

    def split_data_by_periods(self, train_months=12, test_months=3):
        """
        Split both 4H and 1-minute data into walk-forward periods.

        Returns:
            List of (train_4h, test_4h, train_1m, test_1m, period_name) tuples
        """
        data_4h = self.raw_data_4h.sort_values('Date').reset_index(drop=True)
        data_1m = self.raw_data_1m.sort_values('Date').reset_index(drop=True)

        min_date = data_4h['Date'].min()
        max_date = data_4h['Date'].max()

        periods = []
        current_start = min_date
        period_num = 1

        while True:
            # Define training period
            train_start = current_start
            train_end = train_start + pd.DateOffset(months=train_months)

            # Define testing period
            test_start = train_end
            test_end = test_start + pd.DateOffset(months=test_months)

            # Break if test period exceeds data
            if test_end > max_date:
                if test_start < max_date:
                    test_end = max_date

                    # Split 4H data
                    train_4h = data_4h[(data_4h['Date'] >= train_start) &
                                       (data_4h['Date'] < train_end)].copy()
                    test_4h = data_4h[(data_4h['Date'] >= test_start) &
                                      (data_4h['Date'] <= test_end)].copy()

                    # Split 1-minute data
                    train_1m = data_1m[(data_1m['Date'] >= train_start) &
                                       (data_1m['Date'] < train_end)].copy()
                    test_1m = data_1m[(data_1m['Date'] >= test_start) &
                                      (data_1m['Date'] <= test_end)].copy()

                    if len(train_4h) > 0 and len(test_4h) > 0 and len(test_1m) > 0:
                        period_name = f"Period_{period_num}_Train_{train_start.strftime('%Y%m')}_Test_{test_start.strftime('%Y%m')}"
                        periods.append((train_4h, test_4h, train_1m, test_1m, period_name))
                break

            # Extract data for this period
            train_4h = data_4h[(data_4h['Date'] >= train_start) &
                               (data_4h['Date'] < train_end)].copy()
            test_4h = data_4h[(data_4h['Date'] >= test_start) &
                              (data_4h['Date'] < test_end)].copy()

            train_1m = data_1m[(data_1m['Date'] >= train_start) &
                               (data_1m['Date'] < train_end)].copy()
            test_1m = data_1m[(data_1m['Date'] >= test_start) &
                              (data_1m['Date'] < test_end)].copy()

            # Minimum data requirements
            if len(train_4h) > 100 and len(test_4h) > 10 and len(test_1m) > 100:
                period_name = f"Period_{period_num}_Train_{train_start.strftime('%Y%m')}_Test_{test_start.strftime('%Y%m')}"
                periods.append((train_4h, test_4h, train_1m, test_1m, period_name))
                period_num += 1

            # Move forward
            current_start = current_start + pd.DateOffset(months=test_months)

            if current_start + pd.DateOffset(months=train_months) > max_date:
                break

        return periods

    def run_backtest_with_params(self, data_4h, data_1m, params):
        """Run backtest with specific parameters using Numba-compiled function."""
        try:
            # Process 4H data
            data_4h = data_4h.copy()
            data_4h['Date'] = pd.to_datetime(data_4h['Date'])
            data_4h = data_4h.sort_values(['Ticker', 'Date']).reset_index(drop=True)

            # Compute JMA and signals per ticker
            jma_results = []
            for ticker, group in data_4h.groupby('Ticker'):
                jma_group = compute_jma_and_signals(
                    group,
                    smooth_length=int(params['smooth_length']),
                    normalization_period=int(params['normalization_period']),
                    upper_threshold=params['upper_threshold'],
                    lower_threshold=params['lower_threshold'],
                    rs_period=int(params['rs_period']),
                    rs_percentile=int(params['rs_percentile'])
                )
                jma_results.append(jma_group)
            data_4h = pd.concat(jma_results, ignore_index=True)

            # Compute ATR indicators
            atr_calc = ATRIndicator()
            for period in [14, 5]:
                col_name = f'ATR_{period}'
                data_4h[col_name] = 0.0
                for ticker, group in data_4h.groupby('Ticker'):
                    idx = group.index
                    data_4h.loc[idx, col_name] = atr_calc.compute_atr(group, period=period, shift=1)

            # Compute ATR_max
            atr_max_results = []
            for ticker, group in data_4h.groupby('Ticker'):
                atr_max_vals = compute_atr_max(group, params['atr_mult'])
                group_copy = group.copy()
                group_copy['ATR_max'] = atr_max_vals
                atr_max_results.append(group_copy)
            data_4h = pd.concat(atr_max_results, ignore_index=True)

            # Shift signals
            data_4h['Signal'] = data_4h.groupby('Ticker', group_keys=False)['Signal'].shift(1)
            data_4h = data_4h.dropna(subset=['Signal'])
            data_4h['Signal'] = data_4h['Signal'].astype(np.int8)

            # Map signals to 1-minute data
            data_1min = map_signals_to_1min(data_1m, data_4h)
            data_1min = data_1min.sort_values(['Ticker', 'Date']).reset_index(drop=True)

            # Run Numba-compiled backtest
            all_trades = []
            for ticker, group in data_1min.groupby('Ticker'):
                # Convert to numpy arrays for Numba
                dates = group['Date'].values.astype('datetime64[ns]').astype(np.int64)
                opens = group['open'].values.astype(np.float64)
                highs = group['high'].values.astype(np.float64)
                lows = group['low'].values.astype(np.float64)
                closes = group['close'].values.astype(np.float64)
                signals = group['Signal'].values.astype(np.int8)
                atr_maxs = group['ATR_max'].values.astype(np.float64)
                dates_4h = group['Date_4H'].values.astype('datetime64[ns]').astype(np.int64)

                result = backtest_1min_exits_numba(
                    dates, opens, highs, lows, closes, signals, atr_maxs, dates_4h,
                    self.initial_capital, self.fee_rate, self.slippage_rate,
                    params['entry_sl_pct']
                )

                # Unpack results
                (entry_times, exit_times, entry_prices, exit_prices,
                 pnls, cumulative_pnls, directions, entry_4h_times,
                 exit_4h_times, bars_since_entries, exit_reasons) = result

                # Convert back to readable format
                for i in range(len(entry_times)):
                    all_trades.append((
                        ticker,
                        pd.Timestamp(entry_times[i]),
                        pd.Timestamp(exit_times[i]),
                        entry_prices[i],
                        exit_prices[i],
                        pnls[i],
                        "Entry SL (2%)" if exit_reasons[i] == 0 else "Trailing SL",
                        cumulative_pnls[i],
                        "long" if directions[i] == 1 else "short",
                        pd.Timestamp(entry_4h_times[i]),
                        pd.Timestamp(exit_4h_times[i]),
                        bars_since_entries[i]
                    ))

            if len(all_trades) == 0:
                return None, None

            # Create tradebook
            tradebook = pd.DataFrame(all_trades, columns=[
                'Ticker', 'Entry Time', 'Exit Time', 'Entry Price', 'Exit Price',
                'PnL', 'Exit Reason', 'Cumulative PnL', 'Direction',
                'Entry 4H Time', 'Exit 4H Time', 'Bars Since Entry'
            ])

            # Calculate metrics
            metrics = calculate_metrics(tradebook, self.initial_capital)

            return tradebook, metrics

        except Exception as e:
            print(f"    Error in backtest: {str(e)}")
            import traceback
            traceback.print_exc()
            return None, None

    def optimize_on_train_data(self, train_4h, train_1m, param_grid, weights):
        """Run grid search optimization on training data."""
        print(f"    Optimizing on {len(train_4h)} 4H bars, {len(train_1m)} 1min bars...")

        # Generate parameter combinations
        keys = param_grid.keys()
        values = param_grid.values()
        param_combinations = [dict(zip(keys, v)) for v in product(*values)]

        results = []

        # Test each combination
        with tqdm(total=len(param_combinations), desc="    Training", leave=False) as pbar:
            for params in param_combinations:
                _, metrics = self.run_backtest_with_params(train_4h, train_1m, params)
                if metrics is not None and metrics['total_trades'] > 0:
                    result = {**params, **metrics}
                    results.append(result)
                pbar.update(1)

        if len(results) == 0:
            print("    ⚠️  No valid results on training data!")
            return None

        # Calculate composite scores
        results_df = pd.DataFrame(results)
        all_profit_to_drawdown = results_df['profit_to_drawdown'].values
        results_df['composite_score'] = results_df.apply(
            lambda row: calculate_composite_score(row.to_dict(), weights, all_profit_to_drawdown),
            axis=1
        )

        # Get best parameters
        best_params = results_df.sort_values('composite_score', ascending=False).iloc[0]
        best_params_dict = {k: best_params[k] for k in param_grid.keys()}

        print(f"    ✓ Best composite score: {best_params['composite_score']:.4f}")
        print(f"    ✓ Best train PnL: ${best_params['total_pnl']:,.2f}")

        return best_params_dict

    def walk_forward_optimize(self, param_grid, weights=None, train_months=6,
                             test_months=3, save_path=None):
        """
        Run complete walk-forward optimization.

        Args:
            param_grid: Parameter ranges to test
            weights: Metric weights for composite score
            train_months: Months for training period
            test_months: Months for testing period
            save_path: Path to save combined results

        Returns:
            combined_tradebook: All out-of-sample trades
            period_results: Summary of each period's performance
        """

        if weights is None:
            weights = {
                'profit_to_drawdown': 0.35,
                'equity_r2': 0.25,
                'monthly_consistency': 0.25,
                'daily_drawdown': 0.15
            }

        print(f"\n{'='*80}")
        print(f"WALK-FORWARD OPTIMIZATION (Numba-Optimized 1-Minute Exit Strategy)")
        print(f"{'='*80}")
        print(f"Training Period:        {train_months} months")
        print(f"Testing Period:         {test_months} months")
        print(f"4H Data Range:          {self.raw_data_4h['Date'].min()} to {self.raw_data_4h['Date'].max()}")
        print(f"1M Data Range:          {self.raw_data_1m['Date'].min()} to {self.raw_data_1m['Date'].max()}")
        print(f"{'='*80}\n")

        # Split data into periods
        periods = self.split_data_by_periods(train_months, test_months)
        print(f"Created {len(periods)} walk-forward periods\n")

        all_tradebooks = []
        period_summaries = []

        # Process each period
        for i, (train_4h, test_4h, train_1m, test_1m, period_name) in enumerate(periods, 1):
            print(f"\n{'='*80}")
            print(f"PERIOD {i}/{len(periods)}: {period_name}")
            print(f"{'='*80}")
            print(f"Train 4H: {train_4h['Date'].min()} to {train_4h['Date'].max()} ({len(train_4h)} bars)")
            print(f"Test 4H:  {test_4h['Date'].min()} to {test_4h['Date'].max()} ({len(test_4h)} bars)")
            print(f"Test 1M:  {test_1m['Date'].min()} to {test_1m['Date'].max()} ({len(test_1m)} bars)")

            # Optimize on training data
            best_params = self.optimize_on_train_data(train_4h, train_1m, param_grid, weights)

            if best_params is None:
                print(f"⚠️  Skipping period {i} - no valid training results")
                continue

            # Test on out-of-sample data
            print(f"\n    Testing on out-of-sample data...")
            test_tradebook, test_metrics = self.run_backtest_with_params(test_4h, test_1m, best_params)

            if test_tradebook is None:
                print(f"    ⚠️  No trades on test data")
                continue

            # Add period identifier
            test_tradebook['Period'] = period_name
            test_tradebook['Period_Num'] = i

            # Store results
            all_tradebooks.append(test_tradebook)

            period_summary = {
                'period': period_name,
                'period_num': i,
                'train_start': train_4h['Date'].min(),
                'train_end': train_4h['Date'].max(),
                'test_start': test_4h['Date'].min(),
                'test_end': test_4h['Date'].max(),
                **best_params,
                **{f'test_{k}': v for k, v in test_metrics.items()}
            }
            period_summaries.append(period_summary)

            # Print test results
            print(f"\n    OUT-OF-SAMPLE RESULTS:")
            print(f"    Total PnL:              ${test_metrics['total_pnl']:,.2f}")
            print(f"    Profit/Drawdown:        {test_metrics['profit_to_drawdown']:.3f}")
            print(f"    Equity R²:              {test_metrics['equity_r2']:.3f}")
            print(f"    Monthly Consistency:    {test_metrics['monthly_consistency']:.1%}")
            print(f"    Win Rate:               {test_metrics['win_rate']:.1%}")
            print(f"    Total Trades:           {test_metrics['total_trades']}")
            print(f"    Max Drawdown:           ${test_metrics['max_drawdown']:,.2f}")

        # Combine all tradebooks
        if len(all_tradebooks) == 0:
            print("\n⚠️  No valid out-of-sample results!")
            return None, None

        combined_tradebook = pd.concat(all_tradebooks, ignore_index=True)
        combined_tradebook = combined_tradebook.sort_values('Exit Time').reset_index(drop=True)

        # Calculate combined metrics
        combined_metrics = calculate_metrics(combined_tradebook, self.initial_capital)

        # Save results
        if save_path:
            tradebook_path = save_path.replace('.csv', '_tradebook.csv')
            combined_tradebook.to_csv(tradebook_path, index=False)
            print(f"\n✓ Combined tradebook saved to {tradebook_path}")

            summary_path = save_path.replace('.csv', '_period_summary.csv')
            pd.DataFrame(period_summaries).to_csv(summary_path, index=False)
            print(f"✓ Period summary saved to {summary_path}")

        # Print final results
        print(f"\n{'='*80}")
        print(f"WALK-FORWARD OPTIMIZATION COMPLETE")
        print(f"{'='*80}")
        print(f"Total Periods:          {len(periods)}")
        print(f"Successful Periods:     {len(all_tradebooks)}")
        print(f"\nCOMBINED OUT-OF-SAMPLE PERFORMANCE:")
        print(f"Total PnL:              ${combined_metrics['total_pnl']:,.2f}")
        print(f"Profit/Drawdown:        {combined_metrics['profit_to_drawdown']:.3f}")
        print(f"Equity R²:              {combined_metrics['equity_r2']:.3f}")
        print(f"Monthly Consistency:    {combined_metrics['monthly_consistency']:.1%}")
        print(f"Daily Drawdown Score:   {combined_metrics['daily_drawdown']:.3f}")
        print(f"Win Rate:               {combined_metrics['win_rate']:.1%}")
        print(f"Total Trades:           {combined_metrics['total_trades']}")
        print(f"Avg PnL/Trade:          ${combined_metrics['avg_pnl_per_trade']:,.2f}")
        print(f"Max Drawdown:           ${combined_metrics['max_drawdown']:,.2f}")
        print(f"{'='*80}\n")

        return combined_tradebook, pd.DataFrame(period_summaries)


# ===============================
# USAGE EXAMPLE
# ===============================

# Define parameter ranges
param_grid = {
    'smooth_length': [14],
    'normalization_period': [36, 60, 72, 84, 96, 108, 120],
    'upper_threshold': [1.25, 1.5, 1.75, 2.0, 2.25],
    'lower_threshold': [-0.75, -1.0, -1.25, -1.5, -1.75],
    'rs_period': [3, 14, 21, 55],
    'rs_percentile': [10, 30, 50, 70],
    'atr_mult': [0.1],
    'entry_sl_pct': [0.02]
}

# Define metric weights
metric_weights = {
    'profit_to_drawdown': 0.3,
    'equity_r2': 0.2,
    'monthly_consistency': 0.2,
    'daily_drawdown': 0.3
}

# Initialize optimizer (Pass BOTH 4H and 1-minute data)
wf_optimizer = WalkForwardOptimizer(
    data_4h=data,      # Your 4-hour OHLC DataFrame
    data_1m=data_1m,   # Your 1-minute OHLC DataFrame
    initial_capital=100000,
    fee_rate=0.0005,
    slippage_rate=0.0005
)

# Run walk-forward optimization
combined_tradebook, period_summary = wf_optimizer.walk_forward_optimize(
    param_grid=param_grid,
    weights=metric_weights,
    train_months=18,
    test_months=6,
    save_path="/content/drive/MyDrive/Tradebooks/SOLUSDT_Walkforward_Optimization.csv"
)

# Analyze results
if combined_tradebook is not None:
    print("\nPERIOD-BY-PERIOD ANALYSIS:")
    print(period_summary[['period', 'test_total_pnl', 'test_total_trades',
                          'test_win_rate', 'test_max_drawdown']])